In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# 1. Define image transformations (Convert to tensor and normalize)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Download and load the Training Set
train_set = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

# 3. Download and load the Test Set
test_set = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print(f"Successfully loaded {len(train_set)} training images.")
print(f"Successfully loaded {len(test_set)} test images.")

100%|██████████| 170M/170M [21:08<00:00, 134kB/s]


Successfully loaded 50000 training images.
Successfully loaded 10000 test images.


In [ ]:
import torch.nn as nn
import torch.optim as optim

**Build CNN**

In [ ]:
from torch.nn.modules import padding
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layer= nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.fc_layer=nn.Sequential(
        nn.Linear(128*4*4,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )

  def forward(self,x):
    x=self.conv_layer(x)
    x=x.view(x.size(0),-1)
    x=self.fc_layer(x)
    return x

In [ ]:
model = CNN()

In [ ]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

**Training CNN**

In [ ]:
epochs =10
for epoch in range(epochs):
  epoch_training_loss=0.0

  for images,labels in train_loader:
    optimizer.zero_grad()
    output =model.forward(images)
    loss = criterian(output,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss+=loss.item()

  print(f"Epoch {epoch+1}/{epochs} Loss: {epoch_training_loss/len(train_loader)}")

Epoch 1/10 Loss: 1.3865888085968965
Epoch 2/10 Loss: 0.9476142495184603
Epoch 3/10 Loss: 0.7625086308668947
Epoch 4/10 Loss: 0.6339996025118682
Epoch 5/10 Loss: 0.5284698215668159
Epoch 6/10 Loss: 0.43529629821667587
Epoch 7/10 Loss: 0.3567779007393991
Epoch 8/10 Loss: 0.27875378270588264
Epoch 9/10 Loss: 0.21823508940789554
Epoch 10/10 Loss: 0.17366117224468827


In [ ]:
correct =0
total =0

with torch.no_grad():
  for images,labels in test_loader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs.data,1)
    total+=labels.size(0)
    correct +=(predicted==labels).sum().item()
print(f"Acuuracy:{correct/total}")

Acuuracy:0.7478
